# Batch Area Optimization Viewer

Run area optimization for each `N` in `[3, 32]` with `10000` iterations, save history plots and result JSON, then display area-colored meshes (not saved).


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

from coordinate_fileio import save_division_result
from sphere_division_algorithms import run_tension_equalizer, spherical_triangle_areas
from sphere_division_visualization import (
    plot_octant_mesh_from_positions_with_area_color,
    plot_optimizer_history_and_distribution,
)


In [ ]:
N_values = range(3, 32 + 1)
iterations = 10000
lr = 0.2
verbose_every = 500
figure_dir = PROJECT_ROOT / 'figures'
result_dir = PROJECT_ROOT / 'results'

figure_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)

summary = []

for N in N_values:
    print(f'\n===== N={N} / iterations={iterations} =====')

    positions_eq, triangle_keys_eq, hist = run_tension_equalizer(
        N, iterations=iterations, lr=lr, verbose_every=verbose_every
    )
    areas_eq = spherical_triangle_areas(positions_eq, triangle_keys_eq)

    history_path = figure_dir / f'plot_optimizer_history_and_distribution_{N}.svg'
    plot_optimizer_history_and_distribution(
        hist=hist,
        areas_eq=areas_eq,
        n=N,
        save_path=history_path,
    )

    result_path = result_dir / f'division_result_{N}.json'
    save_division_result(result_path, N, positions_eq)

    print(f'saved history figure: {history_path.as_posix()}')
    print(f'saved result json   : {result_path.as_posix()}')

    # Display only (no save)
    plot_octant_mesh_from_positions_with_area_color(
        triangle_keys=triangle_keys_eq,
        positions=positions_eq,
        n=N,
        colormap='bwr',
        save_path=None,
    )

    summary.append((N, float(areas_eq.min()), float(areas_eq.max()), float(areas_eq.std(ddof=0))))

print('\n=== Summary (after optimization) ===')
for N, a_min, a_max, a_std in summary:
    print(f'N={N}: min={a_min:.10f}, max={a_max:.10f}, std={a_std:.10f}')
